# EEG-Emotion2Text Kaggle Pipeline

本 Notebook 只保留简洁调度；核心代码在 `eeg2text_core.py` 与 `eeg2text_train.py`。

支持能力:
- Saveinfo CSV 解析情绪标签
- LLM 文本输入通过 CSV 协议加载
- 断点保存/续训
- 最大训练时长保护（防 Kaggle 强制结束导致丢失）

In [ ]:
# !pip -q install scipy transformers accelerate

In [ ]:
# Kaggle 常用路径初始化 Block
import os
import sys
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

candidate_roots = [
    Path.cwd(),
    KAGGLE_WORKING / 'EEG_emotion2text',
    KAGGLE_INPUT / 'eeg-emotion2text',
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'eeg2text_core.py').exists() and (root / 'eeg2text_train.py').exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('未找到 eeg2text_core.py / eeg2text_train.py，请确认文件已上传到 Kaggle。')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from eeg2text_core import CFG
from eeg2text_train import run_loso

In [ ]:
cfg = CFG(
    data_root='/kaggle/input/seed-vii-eeg-feature',
    saveinfo_dir='/kaggle/input/seed-vii-eeg-feature/Saveinfo',
    text_csv_path='/kaggle/input/seed-vii-eeg-feature/text_protocol.csv',
    work_dir='/kaggle/working/eeg2text_ckpt',

    epochs=20,
    batch_size=128,

    resume=True,
    save_every_n_steps=100,
    max_train_hours=8.8,
    time_buffer_minutes=10
)
cfg

In [ ]:
results = run_loso(cfg, run_all_folds=False)  # 最终实验可改为 True
results

## 文本 CSV 协议

文件示例: `text_protocol.csv`

```csv
trial,emotion,l1_text,l2_text,l3_text
1,happy,The person feels joyful.,Clip-level summary for trial 1.,Long narrative for trial 1.
2,neutral,,,
```

规则:
1. 必须有 `trial` 列 (1-80)。
2. `emotion/l1_text/l2_text/l3_text` 为可选列。
3. 缺失字段会自动回退到默认模板文本。

## 续训方式

1. 使用相同 `work_dir`。
2. 保持 `resume=True`。
3. 再次运行训练单元，自动读取 `latest_*.pt` 继续。